# Part 3 — Multi-Agent Supervisor

**Query:** *"What are the key government revenue streams, and how will the Budget for the Future Energy Fund be supported?"*

**Goal:** a LangGraph supervisor that answers it by delegating to two specialist agents:
- **Revenue Agent:** finds and extracts information on government revenue.
- **Expenditure Agent:** finds and analyses government spending, including specific funds, and adds figures up to the correct total.

## 0. Setup

One new dependency on top of Parts 1–2, pinned in `requirements.txt`: `langgraph==1.2.11`.

**Design decision — LangGraph, pinned explicitly.** The task names LangGraph, and it reuses the LangChain + Gemini stack from Parts 1–2 (`ChatGoogleGenerativeAI`, `@tool`, the same retry settings). The next cell imports it directly, so it is pinned in `requirements.txt` rather than left to arrive as a dependency of `langchain`.

In [ ]:
import json
from pathlib import Path

import pdfplumber
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv(Path(".env"))

PDF_PATH = Path("data/fy2024_analysis_of_revenue_and_expenditure.pdf")
assert PDF_PATH.exists(), f"Document not found at {PDF_PATH}"

# Same parser as Parts 1-2 (chosen in part_1.ipynb §1.4), keyed by printed page number (verified in part_1.ipynb §1.1)
with pdfplumber.open(PDF_PATH) as pdf:
    PAGES: dict[int, str] = {i + 1: (page.extract_text() or "") for i, page in enumerate(pdf.pages)}

MODEL = "gemini-3.1-flash-lite"   # same model as Parts 1-2
RETRY = dict(stop_after_attempt=5, wait_exponential_jitter=True,   # same retry settings as Part 2
             exponential_jitter_params={"initial": 4, "max": 60, "exp_base": 2, "jitter": 2})

llm = ChatGoogleGenerativeAI(model=MODEL, temperature=0)


def context_for(page_numbers: list[int]) -> str:
    """Same as Parts 1-2: render selected pages as tagged blocks so the model can cite page numbers."""
    return "\n\n".join(f'<page number="{n}">\n{PAGES[n].strip()}\n</page>' for n in sorted(page_numbers))


def flat(text: str) -> str:
    """Same as Part 2: collapse line breaks and repeated spaces, so a sentence can be matched across lines."""
    return " ".join(text.split())


print(f"Parsed {len(PAGES)} pages | model: {MODEL}")

Parsed 37 pages | model: gemini-3.1-flash-lite


## 1. Where does the answer live?

Before designing agents, the PDF was read by hand to see which pages hold each half of the query, and whether revenue and spending are kept apart in the document or mixed together. That decides how the agents should reach the document.

**Observations:**
1. **The Future Energy Fund is one figure in three places:** a row of Table 2.1 (p16, `5.00` in $billion, FY2024 column), the p18 sentence ("an initial injection of $5.0 billion to invest in critical infrastructure for the energy transition"), and a row of Table 2.4 (p20, `5,000` in $million). In Table 2.1 it is listed under **Top-ups to Endowment and Trust Funds** (20.35 in FY2024). None of these lines ties the fund to a particular revenue source.
2. **Revenue and spending share pages.** Table 2.1 on p16 holds Operating Revenue items (Corporate Income Tax 28.03, Goods and Services Tax 19.39, Personal Income Tax 18.07 in FY2024), the fund top-ups, the Net Investment Returns Contribution (23.50) and the Overall Fiscal Position (0.78) in one table. "Operating Revenue" and "Top-ups to Endowment" are both mentioned on p8, p13, p16, p24 and p25.
3. **Total Revenue is Operating Revenue plus NIRC** (p9 footnote), so NIRC counts as a revenue stream in addition to the operating revenue.

## 2. Design decisions and assumptions

### 2.1 How the agents reach the document

| Option | What it means |
|---|---|
| Page scoping | Each agent can only open a hand-picked list of pages (revenue pages vs spending pages). |
| **Whole-document tools (chosen)** | Both agents search and read any page; their system prompts are what make them specialists. |

**Design decision — whole-document tools, no page scoping.**
- Observation 2: revenue and spending sit in the same tables. Table 2.1 on p16 would have to be in both agents' lists, and so would p8, p13, p24 and p25.
- Page lists are manual and hardcoded: someone reads the document to pick them, and they break for any question outside those pages.
- The cost is observation 3: an agent will find Corporate Income Tax for several years. Every finding must therefore carry the financial year the document labels it with (§4). **If a run takes a figure from the wrong year, the fix goes into the prompts, not into page scoping.**

### 2.2 Supervisor: own `StateGraph` vs agents as tools

| | **Own `StateGraph` (chosen)** | Agents as tools |
|---|---|---|
| How routing happens | A `supervisor` node returns `Route(reasoning, tasks)` | A tool-calling agent calls `ask_revenue_agent(task)` / `ask_expenditure_agent(task)` |
| Why it routed | `reasoning` is a required field, written before the choice | Only the task text, unless an extra argument is added |
| Trace | One printed step per graph node | The supervisor's message list |
| Control flow | In code: conditional edge, step limit, separate synthesis node | Inside the agent's tool loop |
| Code | More: state, nodes, edges | Less: two tool wrappers |

**Design decision — own `StateGraph`.** The task asks for a clear trace of how the supervisor routes and synthesises. With its own node, every routing decision is a typed object with its reason attached, the graph can be drawn (§5), and synthesis is a separate step that can be checked.

### 2.3 Other decisions

| Decision | Why |
|---|---|
| **Hybrid dispatch: the supervisor sends one agent or both per step** | One agent per step makes independent parts wait for each other. Always sending both runs the expenditure agent on a revenue-only question, and the routing stops being a decision. With `tasks` as a list, independent parts run in parallel (LangGraph `Send`), and a part that needs another agent's figure can still wait for it. |
| **Worker agents built with `create_agent`** | Part 2 already wrote a tool loop by hand. The new part here is the supervisor, so the workers use LangChain's prebuilt tool loop. |
| **`add_numbers` tool, Expenditure Agent only, for sums the document doesn't print** | The task asks this agent to sum to the correct figure. A total the document already states is read, not re-added. When an answer needs a sum the document doesn't give, the tool does it, so the LLM never does arithmetic itself (as in Part 1). §6 checks that such a sum came from the tool. |
| **Agents return a structured `AgentReport`: page, quote, financial year, then value** | Same evidence-first order as Part 1. It lets §6 check every quote against its page, and catch a figure labelled with the wrong year. |
| **Synthesis is its own node and sees only the reports, not the pages** | Every figure in the final answer can then be traced back to an agent's finding. |
| **`MAX_STEPS = 3`: only the first 3 supervisor decisions can send agents** | Room for "both agents", a follow-up, and one more before finishing. A later decision goes straight to synthesis, so a supervisor that keeps re-routing can't loop forever. |

### 2.4 Assumptions

1. **"Key government revenue streams" means FY2024 (Budget 2024, Estimated).** The second half of the query is about a fund established in FY2024, so both halves are read for the same year. "Key" means the largest 5 items. NIRC counts as a revenue stream (observation 3). 


## 3. Tools

Three plain Python functions turned into LangChain tools with `@tool`: the docstring is what the LLM reads. They are tested here with no LLM involved, so a later failure can be pinned on either the tool or the prompt.

In [ ]:
from langchain_core.tools import tool


@tool
# Tool 1 (both agents): find which pages mention a term
def search_pages(keyword: str) -> str:
    """Find every line of the document that contains `keyword` (case-insensitive).

    Returns one 'page N: line' per match. Use short keywords, e.g. 'Future Energy Fund' or 'Corporate Income Tax'.
    """
    hits = [f"page {n}: {line.strip()}"
            for n, text in PAGES.items() for line in text.splitlines() if keyword.lower() in line.lower()]
    return "\n".join(hits) if hits else f"No line contains {keyword!r}. Try a shorter or different keyword."


@tool
# Tool 2 (both agents): read a whole page, so a table row comes with its column headers
def read_page(page: int) -> str:
    """Return the full text of one printed page of the document (pages 1-37).

    Read a page before quoting from it: a table row only makes sense with its column headers.
    """
    if page not in PAGES:
        return f"There is no page {page}. Pages run from 1 to {len(PAGES)}."
    return context_for([page])


@tool
# Tool 3 (Expenditure Agent only): a calculator for sums the document doesn't print
def add_numbers(numbers: list[float]) -> float:
    """Add up a list of numbers and return the total. Use this for every sum - never add numbers yourself."""
    return round(sum(numbers), 6)

In [ ]:
print(search_pages.invoke({"keyword": "Future Energy Fund"}))
print(search_pages.invoke({"keyword": "Solar Levy"}))
print(read_page.invoke({"page": 99}))
print()
print(read_page.invoke({"page": 20}))

# The Table 2.4 rows sit between "($ million)" and "Total"; add them with the tool and compare with the stated total
lines20 = PAGES[20].splitlines()
rows = lines20[lines20.index("($ million)") + 1 : next(i for i, l in enumerate(lines20) if l.startswith("Total"))]
components = [float(row.rsplit(" ", 1)[1].replace(",", "")) for row in rows]
print("\nadd_numbers(Table 2.4 rows) =", add_numbers.invoke({"numbers": components}))

page 16: Future Energy Fund - 5.00
page 18: establish the Future Energy Fund with an initial injection of $5.0 billion to invest
page 20: Future Energy Fund 5,000
No line contains 'Solar Levy'. Try a shorter or different keyword.
There is no page 99. Pages run from 1 to 37.

<page number="20">
Top-ups to Endowment and Trust Funds in FY2024 Table 2.4
Estimated FY2024
($ million)
Goods and Services Tax Voucher Fund 6,000
Future Energy Fund 5,000
Edusave Endowment Fund 2,000
Financial Sector Development Fund 2,000
National Productivity Fund 2,000
National Research Fund 1,800
Progressive Wage Credit Scheme Fund 1,000
Skills Development Fund 500
Public Transport Fund 50
Legal Aid Fund 2
Total 20,352
MINISTRY OF FINANCE 20
</page>

add_numbers(Table 2.4 rows) = 20352.0


**Observations:**
- `search_pages` returns every hit with its page number, so an agent can see all three Future Energy Fund mentions at once.
- A keyword with no match and a page that doesn't exist come back as plain messages saying what to try instead, not as exceptions. An agent can read them and try again.
- `add_numbers` over the ten Table 2.4 rows gives **20352.0**, matching the stated `Total 20,352`.

## 4. The specialist agents

Each agent is `create_agent` with its own system prompt and tools, returning an `AgentReport`.

**Design decision — the prompts only describe the role and the evidence rules.** Nothing about specific pages or financial years goes in (§2.1). The `financial_year` field in every finding is what exposes a wrong-year figure, if one is taken.

In [ ]:
from typing import Literal

from langchain.agents import create_agent
from langchain.agents.middleware import ModelRetryMiddleware
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from pydantic import BaseModel, Field


# ---- What an agent hands back: a list of findings, each carrying its evidence ----
class Finding(BaseModel):
    """One fact from the document. Evidence comes before the value, as in Part 1: the value is read off the quote."""
    source_page: int = Field(description="The printed page number this was read from.")
    quote: str = Field(description="The verbatim sentence or table row it comes from. Copy it exactly.")
    item: str = Field(description="What it is about, e.g. 'Corporate Income Tax' or 'Future Energy Fund top-up'.")
    financial_year: str = Field(description="The financial year and basis as the document labels it, e.g. 'Estimated FY2024' or 'Revised FY2023'.")
    value: float | None = Field(description="The figure as a plain number read from the quote; null if the finding is a statement, not a figure.")
    unit: str | None = Field(description="The unit as the document expresses it, e.g. '$ billion', '$ million', 'percent'.")


class AgentReport(BaseModel):
    findings: list[Finding]
    summary: str = Field(description="A short answer to the task, using only the findings above.")


# ---- System prompts: the role says what each agent covers; the rules are the evidence rules from Parts 1-2 ----
REVENUE_SYSTEM_V1 = """You are the Revenue Agent in a team answering questions about Singapore's Analysis of Revenue and Expenditure, Financial Year 2024 (Ministry of Finance).

Your role: identify and extract information on government REVENUE - tax and non-tax Operating Revenue, and the Net Investment Returns Contribution (NIRC). Leave spending, transfers and funds to the Expenditure Agent.

Rules:
1. Use ONLY the document, through your tools: search_pages to find where something is, then read_page before quoting from that page. Never answer from memory.
2. Every finding needs the page it came from and a verbatim quote of the sentence or table row.
3. Figures in parentheses are negative: (0.35) means -0.35.
4. When asked for key, main or largest items, report the 5 largest by value for the requested financial year, largest first. A total (e.g. Operating Revenue) is not an item: leave it out.
5. Report only what your task asks for."""

EXPENDITURE_SYSTEM_V1 = """You are the Expenditure Agent in a team answering questions about Singapore's Analysis of Revenue and Expenditure, Financial Year 2024 (Ministry of Finance).

Your role: find and analyse information on government SPENDING - total, operating and development expenditure, special transfers, and top-ups to endowment and trust funds, including specific funds. Leave revenue to the Revenue Agent.

Rules:
1. Use ONLY the document, through your tools: search_pages to find where something is, then read_page before quoting from that page. Never answer from memory.
2. Every finding needs the page it came from and a verbatim quote of the sentence or table row.
3. Figures in parentheses are negative: (0.35) means -0.35.
4. Never add numbers yourself. Use add_numbers for every sum and report its result.
5. Report only what your task asks for."""


# ---- Build one agent: Gemini + its tools + the AgentReport it must end with ----
def make_agent(system_prompt: str, tools: list):
    """A tool-calling agent that ends by returning an AgentReport."""
    retry = ModelRetryMiddleware(max_retries=4, initial_delay=4, max_delay=60, on_failure="error")   # ~Part 2's retry settings
    # create_agent runs the tool loop: call tools until done, then return an AgentReport
    return create_agent(llm, tools, system_prompt=system_prompt, response_format=AgentReport, middleware=[retry])

## 5. The supervisor graph

- **State:** `question`, `reports` (every agent's report, appended), `routes` (every supervisor decision, appended), `final_answer`.
- **`supervisor`:** returns `Route(reasoning, tasks)`, where `reasoning` is written before `tasks` so the choice follows from it. `tasks` holds one entry per agent to run: one entry sends one agent, two entries send both, and none means the reports already answer the question.
- **Conditional edge:** one `Send(agent, {"task": ...})` per entry. No tasks goes to `synthesize`, and so does any decision after the first `MAX_STEPS` (its tasks are dropped).
- **Agent nodes:** run the agent on its task, append the report, return to `supervisor`.
- **`synthesize`:** writes the final answer from the reports only.

In [ ]:
import operator
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph
from langgraph.types import Send

# Only the first MAX_STEPS supervisor decisions can send agents; later ones go straight to synthesis
MAX_STEPS = 3
AGENTS = ["revenue_agent", "expenditure_agent"]


# ---- What the supervisor returns each turn: why, then which agent gets which task ----
class AgentTask(BaseModel):
    agent: Literal["revenue_agent", "expenditure_agent"]
    task: str = Field(description="A self-contained instruction for that agent: what to find, and for which financial year.")


class Route(BaseModel):
    reasoning: str = Field(description=(
        "Written before choosing tasks: which parts of the question the reports already answer, which they don't, "
        "which agent covers each missing part, and whether those parts can run together or one needs another's report first."))
    tasks: list[AgentTask] = Field(description="The agents to run next, one task per agent. Empty when the reports answer the whole question.")


# ---- Shared state every node reads and updates ----
# Annotated[..., operator.add] means a node's update is appended to the list, not written over it
class State(TypedDict):
    question: str
    reports: Annotated[list[dict], operator.add]
    routes: Annotated[list[Route], operator.add]
    final_answer: str


# What an agent node receives through Send: only its own task
class AgentInput(TypedDict):
    task: str


# ---- Prompts for the supervisor and the final synthesis ----
SUPERVISOR_SYSTEM_V1 = """You are the supervisor of two specialist agents answering questions about Singapore's Analysis of Revenue and Expenditure, Financial Year 2024 (Ministry of Finance).

Your agents:
- revenue_agent: government revenue - tax and non-tax Operating Revenue, and the Net Investment Returns Contribution.
- expenditure_agent: government spending - expenditure, special transfers, and top-ups to endowment and trust funds, including specific funds. It can add up figures with a calculator tool.

You never read the document yourself. Each turn, decide which agents run next:
1. Split the question into parts and match each part to the agent whose role covers it. Only send an agent if some part needs it.
2. If the parts are independent, send every agent needed in the same turn; they run in parallel.
3. If a task needs a figure from another agent's report, send that other agent first and the second one in a later turn, putting the figure it needs into its task.
4. Write each task so the agent can act on it alone: what to find, and for which financial year.
5. When the reports so far answer every part of the question, return no tasks. Never re-send an agent for something already reported."""

SYNTH_SYSTEM_V1 = """You write the final answer to the user's question from the specialist agents' reports.

Rules:
1. Use ONLY the reports. Do not add any figure, reason or fact that they do not contain.
2. Answer every part of the question, in the order it was asked.
3. Give the page for each figure, e.g. "$28.03 billion (p16)", and the financial year the report gives it.
4. If the reports do not answer a part of the question, say so plainly instead of filling the gap."""


def reports_text(reports: list[dict]) -> str:
    """What the supervisor and the synthesiser see of each report: who, which task, findings and summary."""
    if not reports:
        return "(none yet)"
    return "\n\n".join(json.dumps({"agent": r["agent"], "task": r["task"], **r["report"].model_dump()}, indent=1)
                       for r in reports)


# ---- The graph ----
def build_graph(prompts: dict):
    """Supervisor + two agents + synthesis, wired as in the diagram. The same function serves every prompt version."""
    # The two specialists, plus the model used directly for routing and for the final answer
    agents = {"revenue_agent": make_agent(prompts["revenue"], [search_pages, read_page]),
              "expenditure_agent": make_agent(prompts["expenditure"], [search_pages, read_page, add_numbers])}
    router = llm.with_structured_output(Route).with_retry(**RETRY)
    writer = llm.with_retry(**RETRY)

    # Node: decide which agents run next, given the question and the reports so far
    def supervisor(state: State) -> dict:
        route = router.invoke([SystemMessage(prompts["supervisor"]),
                               HumanMessage(f"Question: {state['question']}\n\nReports so far:\n{reports_text(state['reports'])}")])
        return {"routes": [route]}

    # Conditional edge: turn the latest Route into where the graph goes next
    def dispatch(state: State):
        route = state["routes"][-1]
        # No tasks, or past the step limit -> write the final answer
        if not route.tasks or len(state["routes"]) > MAX_STEPS:
            return "synthesize"
        # One Send per task; two Sends run both agents in parallel
        return [Send(t.agent, {"task": t.task}) for t in route.tasks]

    # Node factory: runs one agent on its task and stores its report plus every message (tool calls included)
    def agent_node(name: str):
        def run(inp: AgentInput) -> dict:
            result = agents[name].invoke({"messages": [HumanMessage(inp["task"])]})
            return {"reports": [{"agent": name, "task": inp["task"],
                                 "report": result["structured_response"], "messages": result["messages"]}]}
        return run

    # Node: write the final answer from the reports only, never the pages
    def synthesize(state: State) -> dict:
        answer = writer.invoke([SystemMessage(prompts["synthesize"]),
                                HumanMessage(f"Question: {state['question']}\n\nReports:\n{reports_text(state['reports'])}")])
        return {"final_answer": answer.text}

    # Wiring, as in the diagram: START -> supervisor -> agent(s) -> supervisor -> ... -> synthesize -> END
    graph = StateGraph(State)
    graph.add_node("supervisor", supervisor)
    # Each agent always hands back to the supervisor
    for name in AGENTS:
        graph.add_node(name, agent_node(name))
        graph.add_edge(name, "supervisor")
    graph.add_node("synthesize", synthesize)
    graph.add_edge(START, "supervisor")
    # The dashed arrows: dispatch picks one agent, both, or synthesize
    graph.add_conditional_edges("supervisor", dispatch, [*AGENTS, "synthesize"])
    graph.add_edge("synthesize", END)
    return graph.compile()


# Build the v1 graph and draw it from the compiled object
PROMPTS_V1 = {"supervisor": SUPERVISOR_SYSTEM_V1, "revenue": REVENUE_SYSTEM_V1,
              "expenditure": EXPENDITURE_SYSTEM_V1, "synthesize": SYNTH_SYSTEM_V1}
graph_v1 = build_graph(PROMPTS_V1)
print(graph_v1.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	supervisor(supervisor)
	revenue_agent(revenue_agent)
	expenditure_agent(expenditure_agent)
	synthesize(synthesize)
	__end__([<p>__end__</p>]):::last
	__start__ --> supervisor;
	expenditure_agent --> supervisor;
	revenue_agent --> supervisor;
	supervisor -.-> expenditure_agent;
	supervisor -.-> revenue_agent;
	supervisor -.-> synthesize;
	synthesize --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



**Observation:** this diagram is printed as Mermaid text from the compiled graph. The dashed arrows (`-.->`) out of `supervisor` are the conditional edge: it can go to either agent, both, or `synthesize`. The solid arrows (`-->`) always run. It has the same nodes and edges as the diagram in `README.md`.

## 6. Trace and checks

`run_query()` streams the graph and prints every step as it happens: each supervisor decision with its reasoning, each agent's tool calls and findings, and the final answer. `check()` then grades the run:

- **routing:** exactly the expected agents were called.
- **facts:** each expected figure appears in the final answer, matched as text (e.g. `5.0 billion` or `5,000`), so a page citation like `p5` can't pass for a figure.
- **sum from the tool:** where a sum is expected, an `add_numbers` call returned it.
- **quotes on cited pages:** every finding's quote appears on the page it cites, ignoring line breaks (as in Part 2).
- **no wrong year:** no finding labels a figure from another year as FY2024.

In [ ]:
# ---- Printing the trace ----
def short(text: str, width: int = 110) -> str:
    text = flat(text)
    return text if len(text) <= width else text[:width] + " ..."


def show_route(step: int, route: Route) -> None:
    print(f"\n[supervisor, decision {step}]")
    print(f"  reasoning: {route.reasoning}")
    if not route.tasks:
        print("  -> no tasks: go to synthesis")
    elif step > MAX_STEPS:
        print(f"  -> step limit reached ({MAX_STEPS}): go to synthesis, tasks dropped")
    else:
        print(f"  -> sends {' + '.join(t.agent for t in route.tasks)}" + (" in parallel" if len(route.tasks) > 1 else ""))
        for t in route.tasks:
            print(f"     {t.agent}: {t.task}")


def show_report(r: dict) -> None:
    print(f"\n[{r['agent']}]")
    for m in r["messages"]:
        if isinstance(m, AIMessage):
            for call in m.tool_calls:
                if call["name"] != "AgentReport":
                    print(f"  calls {call['name']}({', '.join(f'{k}={v!r}' for k, v in call['args'].items())})")
        elif isinstance(m, ToolMessage) and m.name != "AgentReport":
            print(f"    -> {short(str(m.content))}")
    for f in r["report"].findings:
        print(f"  finding: {f.item} | {f.financial_year} | {f.value} {f.unit or ''} | p{f.source_page}")
    print(f"  summary: {r['report'].summary}")


# ---- Run one query, printing each step as the graph produces it ----
def run_query(graph, question: str) -> dict:
    print(f"QUESTION: {question}")
    state, step = None, 0
    for mode, chunk in graph.stream({"question": question, "reports": [], "routes": []},
                                    stream_mode=["updates", "values"]):
        # "values" is the full state after each step (kept for the checks); "updates" is what one node just changed (printed)
        if mode == "values":
            state = chunk
            continue
        for node, update in chunk.items():
            if node == "supervisor":
                step += 1
                show_route(step, update["routes"][-1])
            elif node in AGENTS:
                for r in update["reports"]:
                    show_report(r)
            elif node == "synthesize":
                print(f"\n[synthesize]\n{update['final_answer']}")
    return state


# ---- Grade one finished run against expected values read by hand ----
def check(state: dict, expected: dict) -> dict[str, bool]:
    called = {r["agent"] for r in state["reports"]}
    findings = [f for r in state["reports"] for f in r["report"].findings]
    answer = flat(state["final_answer"])
    # Every result add_numbers returned during the run
    sums = []
    for r in state["reports"]:
        for m in r["messages"]:
            if isinstance(m, ToolMessage) and m.name == "add_numbers":
                try:
                    sums.append(float(m.content))
                except ValueError:
                    pass

    # One PASS/FAIL entry per check
    results = {f"routing: {sorted(expected['agents'])}": called == expected["agents"]}
    for name, values in expected["facts"].items():
        results[f"fact in answer: {name}"] = any(v in answer for v in values)
    if expected.get("sum"):
        results[f"sum from add_numbers: {expected['sum']}"] = any(abs(s - v) < 0.001 for s in sums for v in expected["sum"])
    # A quote must appear on the page it cites (ignoring line breaks)
    off_page = [f for f in findings if flat(f.quote) not in flat(PAGES.get(f.source_page, ""))]
    results["quotes on cited pages"] = not off_page
    # A known value from another year must not be labelled FY2024
    wrong_year = [f for f in findings for item, values in expected.get("not_fy2024", {}).items()
                  if item.lower() in f.item.lower() and "2024" in f.financial_year and f.value in values]
    results["no wrong-year figure labelled FY2024"] = not wrong_year

    # Print the verdicts, then the details behind any FAIL
    print(f"agents called: {sorted(called)} | add_numbers results: {sums}")
    for name, ok in results.items():
        print(f"  {'PASS' if ok else 'FAIL'}  {name}")
    for f in off_page:
        print(f"      quote not found on p{f.source_page}: {f.quote!r}")
    for f in wrong_year:
        print(f"      wrong year: {f.item} {f.value} labelled {f.financial_year!r} (p{f.source_page})")
    return results

## 7. The task query — prompts v1

Reference values, read by hand from the PDF (§1) and from §3's output:

| Part of the query | Expected | Source |
|---|---|---|
| Key revenue streams, FY2024: the 5 largest, no totals | Corporate Income Tax 28.03, NIRC 23.50, Goods and Services Tax 19.39, Personal Income Tax 18.07, Other Taxes 8.86 ($billion); in $million: 28,029, 23,501, 19,394, 18,075, 8,856. The Operating Revenue total (108.64) is not a stream | Table 2.1 p16; Table 3.1a p24, Table 3.2a p26 |
| Future Energy Fund budget | $5.0 billion initial injection (5,000 in $million) Listed under Top-ups to Endowment and Trust Funds| p16, p18, p20 |

In [ ]:
TASK_QUERY = "What are the key government revenue streams, and how will the Budget for the Future Energy Fund be supported?"

state_v1 = run_query(graph_v1, TASK_QUERY)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


QUESTION: What are the key government revenue streams, and how will the Budget for the Future Energy Fund be supported?

[supervisor, decision 1]
  reasoning: I need to identify the key government revenue streams and the funding source for the Future Energy Fund. The revenue_agent covers revenue streams, and the expenditure_agent covers the funding of specific funds like the Future Energy Fund. These tasks are independent and can be performed in parallel.
  -> sends revenue_agent + expenditure_agent in parallel
     revenue_agent: Identify the key government revenue streams for Financial Year 2024, including tax and non-tax Operating Revenue and the Net Investment Returns Contribution.
     expenditure_agent: Identify how the Budget for the Future Energy Fund will be supported in Financial Year 2024.



[expenditure_agent]
  calls search_pages(keyword='Future Energy Fund')
    -> page 16: Future Energy Fund - 5.00 page 18: establish the Future Energy Fund with an initial injection of $5.0 ...
  calls read_page(page=18)
    -> <page number="18"> 2.6 Special Transfers Special Transfers to Households ($2.6 billion) In addition to transfe ...
  calls read_page(page=16)
    -> <page number="16"> Budget for FY2024 Table 2.1 Revised E s t i m ated C hange Over BLANK FY2023 FY2024 Revised ...
  finding: Future Energy Fund establishment | Budget 2024 | 5.0 $ billion | p18
  finding: Future Energy Fund top-up | Estimated FY2024 | 5.0 $ billion | p16
  summary: In Financial Year 2024, the Government will establish the Future Energy Fund with an initial injection of $5.0 billion to invest in critical infrastructure for the energy transition.

[revenue_agent]
  calls search_pages(keyword='FY2024')
    -> page 2: highlights of the FY2024 Revenue and Expenditure page 3: 2.1 Budget for FY2024 13 pag

In [ ]:
EXPECTED_TASK = {
    "agents": {"revenue_agent", "expenditure_agent"},
    "facts": {                     # FY2024: $ billion (Table 2.1, p16) or $ million (Table 3.1a p24, Table 3.2a p26)
        "Corporate Income Tax 28.03":    ["28.03", "28.0 billion", "28,029"],
        "Goods and Services Tax 19.39":  ["19.39", "19.4 billion", "19,394"],
        "Personal Income Tax 18.07":     ["18.07", "18.1 billion", "18,075"],
        "NIRC 23.5":                     ["23.5", "23,501"],
        "Other Taxes 8.86":              ["8.86", "8.9 billion", "8,856"],
        "Future Energy Fund 5.0 billion": ["5.0 billion", "5.00 billion", "5,000"],
    },
    "not_fy2024": {"Corporate Income Tax": [28.38, 28.4, 28380]},   # Revised FY2023 values
}

check_v1 = check(state_v1, EXPECTED_TASK)

agents called: ['expenditure_agent', 'revenue_agent'] | add_numbers results: []
  PASS  routing: ['expenditure_agent', 'revenue_agent']
  PASS  fact in answer: Corporate Income Tax 28.03
  PASS  fact in answer: Goods and Services Tax 19.39
  PASS  fact in answer: Personal Income Tax 18.07
  PASS  fact in answer: NIRC 23.5
  PASS  fact in answer: Other Taxes 8.86
  PASS  fact in answer: Future Energy Fund 5.0 billion
  PASS  quotes on cited pages
  PASS  no wrong-year figure labelled FY2024


**Observations:**
- **Routing:** decision 1 sent both agents in parallel, and decision 2 finished. All 9 checks pass.
- **"Key" was applied.** The Revenue Agent returned exactly five findings, largest first: Corporate Income Tax 28,029, NIRC 23,501, Goods and Services Tax 19,394, Personal Income Tax 18,075 and Other Taxes 8,856 ($ million, Estimated FY2024). The Operating Revenue total is not among them.
- **It answered from the Statistical Annex, not Table 2.1.** After searching "FY2024" it read Table 3.2a (p26) and Table 3.1a (p24), which print the same FY2024 figures in $ million, so the expected values accept both units. It read five pages to get there, p26 twice.
- **No sum was needed, and none was made.** `add_numbers results: []`. Nothing in this query has several values to add up, which is what the Expenditure Agent's rule 4 is for.
- **The Future Energy Fund half restates the amount.** The Expenditure Agent's two findings are the same $5.0 billion, from p18 and from its Table 2.1 row on p16. The answer says the fund "will be supported ... by an initial injection of $5.0 billion ... to invest in critical infrastructure for the energy transition". Decision 2 judged that part answered.

## 8. Three more queries

Same `graph_v1`, `run_query()` and `check()` as §7. Each query needs a different routing:

| Query | Needs | Expected routing | Expected answer |
|---|---|---|---|
| 1. GST estimate for FY2024 and why it rises | revenue only | `revenue_agent` alone | $19.4 billion (Estimated FY2024), due to the increase in the GST rate and expected growth in consumption (p13) |
| 2. Future Energy Fund and Financial Sector Development Fund top-ups combined | spending only, with a sum the document doesn't print | `expenditure_agent` alone, using `add_numbers` | $5.0 billion + $2.0 billion = **$7.0 billion** (5,000 + 2,000 = 7,000 in $million) (Table 2.1 p16, Table 2.4 p20) |
| 3. Could the GST increase alone cover the Future Energy Fund injection | a figure from each agent, then a comparison | both agents, combined by the supervisor and synthesis | **No:** the GST increase is $3.03 billion (Table 2.1, p16), less than the $5.0 billion injection (p18) |

### 8.1 Revenue only

In [ ]:
Q1 = "What is the estimated Goods and Services Tax collection for FY2024, and why is it expected to increase?"

state_q1 = run_query(graph_v1, Q1)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


QUESTION: What is the estimated Goods and Services Tax collection for FY2024, and why is it expected to increase?

[supervisor, decision 1]
  reasoning: The user is asking for the estimated Goods and Services Tax (GST) collection for FY2024 and the reasons for its expected increase. This information falls under the scope of the revenue_agent.
  -> sends revenue_agent
     revenue_agent: Find the estimated Goods and Services Tax (GST) collection for FY2024 and the reasons provided for its expected increase.



[revenue_agent]
  calls search_pages(keyword='Goods and Services Tax')
    -> page 5: collections from the Goods and Services Tax. page 6: Goods and Services Tax collections are revised to ...
  calls read_page(page=13)
    -> <page number="13"> 02 Outlook for Financial Year 2024 2.1 Budget for FY2024 A basic deficit of $6.1 billion (0 ...
  calls read_page(page=16)
    -> <page number="16"> Budget for FY2024 Table 2.1 Revised E s t i m ated C hange Over BLANK FY2023 FY2024 Revised ...
  finding: Goods and Services Tax | Estimated FY2024 | 19.4 $ billion | p13
  summary: The estimated Goods and Services Tax (GST) collection for FY2024 is $19.4 billion. This increase is attributed to the rise in the GST rate and expected growth in consumption.

[supervisor, decision 2]
  reasoning: The revenue_agent has already provided the estimated GST collection for FY2024 and the reasons for the increase. No further information is required.
  -> no tasks: go to synthesis

[synthesize]
The estimated

In [ ]:
EXPECTED_Q1 = {
    "agents": {"revenue_agent"},
    "facts": {                                           # p13 sentence above; Table 2.1 p16
        "GST 19.4 billion":           ["19.39", "19.4 billion"],
        "reason: GST rate increase":  ["GST rate"],
        "reason: consumption growth": ["consumption"],
    },
    "not_fy2024": {"Goods and Services Tax": [16.36, 16.4, 16363]},   # Revised FY2023 values
}

check_q1 = check(state_q1, EXPECTED_Q1)

agents called: ['revenue_agent'] | add_numbers results: []
  PASS  routing: ['revenue_agent']
  PASS  fact in answer: GST 19.4 billion
  PASS  fact in answer: reason: GST rate increase
  PASS  fact in answer: reason: consumption growth
  PASS  quotes on cited pages
  PASS  no wrong-year figure labelled FY2024


**Observations:**
- **Routed to the Revenue Agent alone**, and decision 2 finished. All checks pass.
- The figure and both reasons come from p13, which the agent read (`read_page(page=13)` in the trace). The agent took $19.4 billion from that sentence on p13, labelled Estimated FY2024, and the Revised FY2023 value 16.36 is not used.

### 8.2 Expenditure only, with a sum the document doesn't print

The Future Energy Fund and Financial Sector Development Fund top-ups (5,000 and 2,000 in Table 2.4, p20) have no combined figure anywhere in the document (searching the PDF for 7,000 finds nothing). This is the case the Expenditure Agent's rule 4 is for: several values that have to be added up.

In [ ]:
Q2 = "How much will the Government put into the Future Energy Fund and the Financial Sector Development Fund combined in FY2024?"

state_q2 = run_query(graph_v1, Q2)

QUESTION: How much will the Government put into the Future Energy Fund and the Financial Sector Development Fund combined in FY2024?

[supervisor, decision 1]
  reasoning: The user is asking for the combined top-ups to two specific funds (Future Energy Fund and Financial Sector Development Fund) in FY2024. Both of these fall under the scope of the expenditure_agent. I will task the expenditure_agent to find the specific amounts for both funds for FY2024 and calculate the total.
  -> sends expenditure_agent
     expenditure_agent: Find the amount of government top-ups to the Future Energy Fund and the Financial Sector Development Fund for FY2024, and calculate their combined total.

[expenditure_agent]
  calls search_pages(keyword='Future Energy Fund')
    -> page 16: Future Energy Fund - 5.00 page 18: establish the Future Energy Fund with an initial injection of $5.0 ...
  calls search_pages(keyword='Financial Sector Development Fund')
    -> page 16: Financial Sector Development Fund 

In [ ]:
EXPECTED_Q2 = {
    "agents": {"expenditure_agent"},
    "facts": {                                                # Table 2.4, p20 (§3)
        "Future Energy Fund 5,000":                ["5,000", "5.0 billion", "5.00 billion"],
        "Financial Sector Development Fund 2,000": ["2,000", "2.0 billion", "2.00 billion"],
        "combined 7,000":                          ["7,000", "7.0 billion", "7.00 billion"],
    },
    "sum": [7000, 7.0],                                       # not printed in the document, so it must come from add_numbers
}

check_q2 = check(state_q2, EXPECTED_Q2)

agents called: ['expenditure_agent'] | add_numbers results: [7.0]
  PASS  routing: ['expenditure_agent']
  PASS  fact in answer: Future Energy Fund 5,000
  PASS  fact in answer: Financial Sector Development Fund 2,000
  PASS  fact in answer: combined 7,000
  PASS  sum from add_numbers: [7000, 7.0]
  PASS  quotes on cited pages
  PASS  no wrong-year figure labelled FY2024


**Observations:**
- **Routed to the Expenditure Agent alone**, and decision 2 finished. All checks pass.
- **Rule 4 used as intended.** The agent searched for each fund, read Table 2.1 on p16, and passed the two top-ups to `add_numbers` in that table's own unit, `[5, 2]` ($ billion). The tool returned 7.0, the combined figure the document doesn't print. The supervisor's task said "calculate their combined total", and the calculation still went to the tool.
- **The $7.0 billion total has no finding of its own.** It appears in the agent's summary and the final answer, while the findings hold only the two quoted figures, since a sum has no verbatim quote to cite. The sum check is what confirms it came from `add_numbers`.

### 8.3 Both agents, combined

In [ ]:
Q3 = "Could the expected FY2024 increase in Goods and Services Tax collections alone cover the Future Energy Fund's initial injection?"

state_q3 = run_query(graph_v1, Q3)

QUESTION: Could the expected FY2024 increase in Goods and Services Tax collections alone cover the Future Energy Fund's initial injection?

[supervisor, decision 1]
  reasoning: To answer the question, I need to determine the expected increase in GST collections for FY2024 compared to the previous year (revenue_agent) and the amount of the initial injection into the Future Energy Fund (expenditure_agent). These tasks are independent and can be performed in parallel.
  -> sends revenue_agent + expenditure_agent in parallel
     revenue_agent: Find the expected increase in Goods and Services Tax (GST) collections for FY2024 compared to FY2023.
     expenditure_agent: Find the amount of the initial injection into the Future Energy Fund for FY2024.

[revenue_agent]
  calls search_pages(keyword='Goods and Services Tax')
    -> page 5: collections from the Goods and Services Tax. page 6: Goods and Services Tax collections are revised to ...
  calls read_page(page=13)
    -> <page number="13"

In [ ]:
EXPECTED_Q3 = {
    "agents": {"revenue_agent", "expenditure_agent"},
    "facts": {                                           # p13 sentence above / Table 2.1 p16; p18
        "GST increase 3.0 billion":       ["3.0 billion", "3.03"],
        "Future Energy Fund 5.0 billion": ["5.0 billion", "5.00 billion", "5,000"],
    },
    "not_fy2024": {"Goods and Services Tax": [16.36, 16.4, 16363]},
}

check_q3 = check(state_q3, EXPECTED_Q3)

agents called: ['expenditure_agent', 'revenue_agent'] | add_numbers results: []
  PASS  routing: ['expenditure_agent', 'revenue_agent']
  PASS  fact in answer: GST increase 3.0 billion
  PASS  fact in answer: Future Energy Fund 5.0 billion
  PASS  quotes on cited pages
  PASS  no wrong-year figure labelled FY2024


**Observations:**
- **Both agents were sent in parallel**, then decision 2 finished. The two figures don't depend on each other, so neither task had to wait for the other's report. **None of the four queries used the sequential path** of the hybrid dispatch (§2.3), where one agent's task carries a figure from the other's report.
- **The $3.03 billion increase is a finding with a quote**, read from Table 2.1's change column on p16, and that quote is on its page. The finding's item is just "Goods and Services Tax", the same name the GST level would have, so only the summary says it is the increase.
- No sum was needed and `add_numbers` wasn't called. The comparison, $3.03 billion against $5.0 billion, so "cannot cover", is made by the synthesis step from the two reported figures.

## 9. Summary

**Design decision — this table is filled in by hand from the printed outputs above,** not computed by a cell. §7 and §8 were run in separate sessions, so their results aren't held in memory together.

All four runs use the v1 prompts.

| Query | Supervisor decisions | `add_numbers` calls | Checks passed |
|---|---|---|---|
| Task query | revenue + expenditure → finish | 0 | 9/9 |
| 8.1 revenue only | revenue → finish | 0 | 6/6 |
| 8.2 expenditure only, unstated sum | expenditure → finish | 1 | 7/7 |
| 8.3 both, combined | revenue + expenditure → finish | 0 | 5/5 |

## 10. Limitations

**Retrieval only matches the document's exact wording.** `search_pages` finds lines that contain the keyword as written (ignoring case). A query that uses a phrase with the same meaning but different words isn't caught: "corporate tax" matches no line in the document, while "Corporate Income Tax" appears on 7 pages. An agent can try again with another keyword when a search comes back empty (§3), but only if it guesses the document's own wording.

**In a production system**, this would be replaced by semantic search, i.e. classical RAG: split the document into chunks, embed each chunk, and retrieve the chunks whose embeddings are closest to the query's embedding. That finds relevant passages by meaning, even when the query's words differ from the document's.

**Design decision — why this notebook uses keyword search instead of RAG:**
- **Tables need their headers.** Most answers sit in tables such as Table 2.1, whose column headers are several lines above the rows, so fixed-size chunks would separate a row from the year it belongs to. Chunking strategies can fix that (one chunk per page, a table kept as one chunk, rows rewritten with their headers, or matching a small chunk and returning its whole page), but the table-aware ones need the table's structure, which pdfplumber's text doesn't keep (Part 1 §1.3–1.4). One chunk per page works, and `read_page` already gives the agent exactly that.
- **One document, and queries in its own terms.** It is a single 37-page PDF, and the queries in §7–§8 name things the way the document does ("Future Energy Fund", "Goods and Services Tax"). Keyword search found the right pages: every query passed its checks, including quotes on cited pages (§9).
- **Retrieval you can check in the trace.** Each step shows the exact keyword searched and the page opened, e.g. `search_pages('Future Energy Fund')` then `read_page(18)`, so a wrong page is easy to spot.
- **Less to build and justify.** RAG adds an embedding model, a vector store, a chunking strategy (size, overlap, how tables are split) and a choice of how many chunks to retrieve. Two plain Python tools needed none of that for this task.